# Predicting Peak Electricity Demand in Massachusetts
## Part 1 of 3: Data Sourcing, Extraction & Cleaning

This notebook pulls real hourly data from three public APIs (ISO-NE electricity demand,
Boston weather, and ISO-NE renewable generation), cleans it, aligns all sources to a single
consistent timezone, engineers basic calendar features, and saves a combined dataset for use
in the EDA and modeling notebooks.

**Data sources:**
- **Electricity demand**: EIA Hourly Electric Grid Monitor API (ISO-NE, hourly)
- **Weather**: Open-Meteo Historical Weather API (Boston, MA, hourly)
- **Renewable generation**: EIA Electricity Data API (solar + wind, ISO-NE, hourly)

> **Note on data provenance:** An earlier version of this project used a Kaggle dataset that,
> upon validation (checking temperature-by-month, load-by-hour, autocorrelation, and facility
> coordinates), was found to be synthetically generated with no real temporal or spatial
> structure. It was replaced with the real sources above.

## 1.1 Setup

In [ ]:
import requests
import pandas as pd
import numpy as np

# Get a free API key at https://www.eia.gov/opendata/register.php
API_KEY = "YOUR_EIA_API_KEY_HERE"

## 1.2 Pull Electricity Demand (EIA)

Hourly demand for the ISO-NE balancing authority. EIA returns timestamps as naive datetimes
representing **UTC** — this matters later when we align it with weather data.

In [ ]:
def get_eia_data(start_date, end_date, api_key, offset=0, length=5000):
    url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"
    params = {
        "api_key": api_key,
        "frequency": "hourly",
        "data[0]": "value",
        "facets[respondent][]": "ISNE",
        "facets[type][]": "D",
        "start": start_date,
        "end": end_date,
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "offset": offset,
        "length": length
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

all_records = []
offset = 0
start_date = "2021-01-01T00"
end_date = "2023-12-31T23"

while True:
    data = get_eia_data(start_date, end_date, API_KEY, offset=offset)
    records = data["response"]["data"]
    if not records:
        break
    all_records.extend(records)
    offset += 5000
    print(f"Pulled {len(all_records)} rows so far...")
    if len(records) < 5000:
        break

df = pd.DataFrame(all_records)
df["period"] = pd.to_datetime(df["period"])  # naive UTC
df = df.sort_values("period").set_index("period")
df = df.rename(columns={"value": "Demand (MW)"})

print(df.shape)
df[["Demand (MW)"]].head(10)

## 1.3 Clean Demand Data

In [ ]:
df["Demand (MW)"] = pd.to_numeric(df["Demand (MW)"], errors="coerce")
print("Missing values:", df["Demand (MW)"].isna().sum())

df = df.dropna(subset=["Demand (MW)"])
print("Shape after dropping missing values:", df.shape)

## 1.4 Pull Weather Data (Open-Meteo)

Requested in **UTC** (not local Eastern time) so it aligns directly with the demand data's
native timezone, avoiding a timestamp mismatch when the two are joined.

In [ ]:
LAT, LON = 42.3601, -71.0589  # Boston, MA

def get_weather_data(start_date, end_date, lat=LAT, lon=LON):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,shortwave_radiation",
        "timezone": "UTC"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

weather_data = get_weather_data("2021-01-01", "2023-12-31")

weather_df = pd.DataFrame({
    "period": weather_data["hourly"]["time"],
    "Temperature (°C)": weather_data["hourly"]["temperature_2m"],
    "Humidity (%)": weather_data["hourly"]["relative_humidity_2m"],
    "Wind Speed (m/s)": weather_data["hourly"]["wind_speed_10m"],
    "Solar Radiation (W/m²)": weather_data["hourly"]["shortwave_radiation"]
})
weather_df["period"] = pd.to_datetime(weather_df["period"])  # naive UTC
weather_df = weather_df.set_index("period")

print(weather_df.shape)
weather_df.head()

## 1.5 Pull Renewable Generation Data (EIA)

Hourly solar and wind generation for ISO-NE, also returned in UTC by the same API.

In [ ]:
def get_eia_fuel_data(start_date, end_date, api_key, fueltype, offset=0, length=5000):
    url = "https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/"
    params = {
        "api_key": api_key,
        "frequency": "hourly",
        "data[0]": "value",
        "facets[respondent][]": "ISNE",
        "facets[fueltype][]": fueltype,   # "SUN" or "WND"
        "start": start_date,
        "end": end_date,
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "offset": offset,
        "length": length
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

def pull_full_fuel_series(fueltype, start_date, end_date, api_key):
    all_records = []
    offset = 0
    while True:
        data = get_eia_fuel_data(start_date, end_date, api_key, fueltype, offset=offset)
        records = data["response"]["data"]
        if not records:
            break
        all_records.extend(records)
        offset += 5000
        if len(records) < 5000:
            break
    return pd.DataFrame(all_records)

solar_df = pull_full_fuel_series("SUN", "2021-01-01T00", "2023-12-31T23", API_KEY)
wind_df = pull_full_fuel_series("WND", "2021-01-01T00", "2023-12-31T23", API_KEY)

print(solar_df.shape, wind_df.shape)
solar_df.head()

In [ ]:
def clean_fuel_df(fuel_df, colname):
    fuel_df = fuel_df.copy()
    fuel_df["period"] = pd.to_datetime(fuel_df["period"])  # naive UTC
    fuel_df["value"] = pd.to_numeric(fuel_df["value"], errors="coerce")
    fuel_df = fuel_df.set_index("period")[["value"]].rename(columns={"value": colname})
    return fuel_df

solar_clean = clean_fuel_df(solar_df, "Solar Generation (MWh)")
wind_clean = clean_fuel_df(wind_df, "Wind Generation (MWh)")

## 1.6 Combine All Sources & Fix Timezone (once, correctly)

**This is the most important cell in the notebook.** All three sources (demand, weather,
generation) are naive-UTC at this point, so they join correctly on raw timestamps.
The single timezone conversion to US/Eastern — and every calendar feature derived from it —
happens **after** the join, in one place, so nothing downstream is silently computed from
the wrong timezone.

In [ ]:
# Join all three sources while everything is still naive UTC
df_combined = df.join(weather_df, how="inner")
df_combined = df_combined.join(solar_clean, how="left").join(wind_clean, how="left")

# Convert to US/Eastern ONCE, after all joins are done
df_combined.index = df_combined.index.tz_localize("UTC").tz_convert("America/New_York")

# Derive ALL calendar features fresh, from the now-correctly-localized index
df_combined["hour"] = df_combined.index.hour
df_combined["dayofweek"] = df_combined.index.dayofweek
df_combined["month"] = df_combined.index.month
df_combined["is_weekend"] = df_combined["dayofweek"].isin([5, 6]).astype(int)

print(df_combined.shape)
df_combined[["Demand (MW)", "Temperature (°C)", "Solar Generation (MWh)", "Wind Generation (MWh)"]].head(10)

## 1.7 Add Holiday Flag

In [ ]:
import sys
!{sys.executable} -m pip install holidays --quiet

import holidays

us_holidays = holidays.US(years=[2020, 2021, 2022, 2023])
df_combined["is_holiday"] = df_combined.index.date
df_combined["is_holiday"] = df_combined["is_holiday"].apply(lambda d: d in us_holidays).astype(int)

print(df_combined["is_holiday"].value_counts())

## 1.8 Validate the Cleaned Dataset

Quick checks to confirm the join and timezone fix worked correctly before saving —
each of these should show a real, physically sensible pattern.

In [ ]:
# dtypes should all be numeric (int32/int64), not boolean
print(df_combined[["hour", "dayofweek", "month", "is_weekend"]].dtypes)
print()
print("Weekend hour count (should be ~28.7% of total):")
print(df_combined["is_weekend"].value_counts(normalize=True))

In [ ]:
# Temperature should show a real winter/summer contrast (not flat ~20°C every month)
print("Avg temperature by month:")
print(df_combined.groupby("month")["Temperature (°C)"].mean())

In [ ]:
# Solar generation should peak at midday (hour 12-13), not at night
print("Avg solar generation by hour (should peak midday):")
print(df_combined.groupby("hour")["Solar Generation (MWh)"].mean())

## 1.9 Save the Combined, Cleaned Dataset

In [ ]:
df_combined.to_csv("isone_full_dataset.csv")
print("Saved:", df_combined.shape)